# Document Classifier Training (SVM + RNN)

Notebook version of `doc_clf.train` for document-level classification.

## Steps
1. Run the path setup cell.
2. Configure input paths and hyperparameters.
3. Run the training cell.

In [ ]:
import sys
from pathlib import Path

# Ensure project src/ is on the import path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Using project root: {project_root}")
print(f"Added to sys.path: {src_path}")

In [ ]:
from pathlib import Path

# === Configuration ===
NEWS_TRAIN = Path("../src/data/news_train.jsonl")
NEWS_TEST = Path("../src/data/news_test.jsonl")
MCC_CHECKPOINT = Path("../src/checkpoints/mcc_bert-base-uncased.pt")
MODEL_NAME = "bert-base-uncased"
EPOCHS = 3
BATCH_SIZE = 4
LR = 1e-3

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from mcc.data import LABEL2ID, MCC_LABELS
from doc_clf.features import extract_features_for_corpus
from doc_clf.models import ArgumentRNNClassifier, ArgumentRNNConfig, compute_macro_f1, train_svm
from doc_clf.train import SequenceDataset, collate_sequences, label_to_int, train_rnn, evaluate_rnn

In [ ]:
# === Training ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Training data: {NEWS_TRAIN}")
print(f"Test data: {NEWS_TEST}")
print(f"MCC checkpoint: {MCC_CHECKPOINT}\n")

print("Extracting features from training data...")
ratio_features, sequences, labels = extract_features_for_corpus(
    NEWS_TRAIN, MCC_CHECKPOINT, MODEL_NAME, device
)
print("\nExtracting features from test data...")
ratio_features_test, sequences_test, labels_test = extract_features_for_corpus(
    NEWS_TEST, MCC_CHECKPOINT, MODEL_NAME, device
)
print()

# Train SVM on ratios
print("\n" + "="*50)
print("Training SVM Classifier")
print("="*50)
feature_matrix = np.array([[feat[f"ratio_{label}"] for label in MCC_LABELS] for feat in ratio_features])
svm_labels = [label_to_int(label) for label in labels]
print(f"Training samples: {len(svm_labels)}")
svm_model, svm_train_f1 = train_svm(feature_matrix, svm_labels)
feature_matrix_test = np.array(
    [[feat[f"ratio_{label}"] for label in MCC_LABELS] for feat in ratio_features_test]
)
svm_labels_test = [label_to_int(label) for label in labels_test]
print(f"Test samples: {len(svm_labels_test)}")
svm_preds = svm_model.predict(feature_matrix_test)
svm_test_f1 = compute_macro_f1(svm_labels_test, svm_preds.tolist())
print(f"\n✓ SVM Results: train F1={svm_train_f1:.3f} | test F1={svm_test_f1:.3f}")

# Train RNN on sequences
print("\n" + "="*50)
print("Training RNN Classifier")
print("="*50)
train_dataset = SequenceDataset(sequences, svm_labels)
test_dataset = SequenceDataset(sequences_test, svm_labels_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_sequences)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_sequences)

rnn_config = ArgumentRNNConfig(vocab_size=len(LABEL2ID))
rnn_model = ArgumentRNNClassifier(rnn_config).to(device)
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=LR)
print(f"Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | Learning rate: {LR}\n")

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    train_rnn(train_loader, rnn_model, optimizer, device)
    f1 = evaluate_rnn(test_loader, rnn_model, device)
    print(f"  ✓ Test F1: {f1:.3f}\n")

print("\n" + "="*50)
print("Training Complete!")
print("="*50)